In [3]:
import sqlite3
import pandas as pd
# Database path - update this if your database is in a different location
DB_PATH = 'meme_evaluations.db'

# Connect to the database
def get_db_connection():
    """Get a connection to the SQLite database"""
    return sqlite3.connect(DB_PATH)

# 1. Session Summary
# =================
# Let's look at a summary of all evaluation sessions

def get_sessions():
    """Get all sessions from the database"""
    conn = get_db_connection()
    sessions_df = pd.read_sql_query("SELECT * FROM sessions ORDER BY timestamp DESC", conn)
    conn.close()
    return sessions_df

df = get_sessions()
df.head()

,id,name,timestamp,model
0,2,Baseline,2025-03-22T06:39:32.821437,gemini-1.5-flash
1,1,Baseline,2025-03-22T06:37:54.524915,gemini-1.5-flash


In [4]:
def get_session_summary(session_id):
    """Get summary statistics for a specific session"""
    conn = get_db_connection()
    
    # Get session info
    session_info = pd.read_sql_query(
        "SELECT * FROM sessions WHERE id = ?", 
        conn, 
        params=[session_id]
    )
    
    # Get evaluation metrics
    metrics = pd.read_sql_query("""
        SELECT 
            COUNT(*) as total_memes,
            SUM(is_humor) as humor_count,
            SUM(is_creative) as creative_count,
            SUM(is_shareable) as shareable_count,
            COUNT(DISTINCT meme_template_id) as unique_templates
        FROM evaluations 
        WHERE session_id = ?
    """, conn, params=[session_id])
    
    conn.close()
    
    if len(session_info) == 0:
        print(f"No session found with ID {session_id}")
        return None, None
    
    return session_info, metrics

session_info, metrics = get_session_summary('2')
session_info

,id,name,timestamp,model
0,2,Baseline,2025-03-22T06:39:32.821437,gemini-1.5-flash


In [5]:
metrics

,total_memes,humor_count,creative_count,shareable_count,unique_templates
0,29,9,9,7,4


In [8]:
conn = get_db_connection()
pd.read_sql_query("select * from evaluations where session_id=2", conn)

,id,session_id,prompt,meme_template_id,gemini_json,imgflip_url,is_humor,is_creative,is_shareable,comments,error,timestamp
0,1,2,"I am trying to make a funny meme, but my sense...",181913649,"{""template_id"": ""181913649"", ""text0"": ""Trying ...",https://i.imgflip.com/9o6wh3.jpg,0,0,0,Variation 1 selected. Drake - Third attempt fa...,None,2025-03-22T06:41:13.790113
1,2,2,"I am trying to make a funny meme using AI, but...",129242436,"{""template_id"": ""129242436"", ""text0"": ""AI can ...",https://i.imgflip.com/9o6wqj.jpg,1,0,1,Variation 2 selected. Variation 3 same as vari...,None,2025-03-22T06:41:58.316178
2,3,2,Pharmacy overlord,129242436,"{""template_id"": ""129242436"", ""text0"": ""Pharmac...",https://i.imgflip.com/9o6wtb.jpg,1,0,1,Variation 1 selected. Variation 3 again failed...,None,2025-03-22T06:42:37.353064
3,4,2,Deep seek a Chinese AI model after being asked...,181913649,"{""template_id"": ""181913649"", ""text0"": ""Asking ...",https://i.imgflip.com/9o6wwb.jpg,0,0,0,"Variation 1 selected. Got stuck on Drake, atte...",None,2025-03-22T06:43:21.728113
4,5,2,Yoda,181913649,"{""template_id"": ""181913649"", ""text0"": ""Using t...",https://i.imgflip.com/9o6wzq.jpg,1,1,1,Variation 2 selected. Good variation,None,2025-03-22T06:43:59.809812
5,6,2,Prompt engineering vs fine-tuning,181913649,"{""template_id"": ""181913649"", ""text0"": ""Prompt ...",https://i.imgflip.com/9o6x2o.jpg,0,1,0,Variation 1 selected. Third variation failed,None,2025-03-22T06:44:26.198063
6,7,2,Please create me a funny meme about parents an...,181913649,"{""template_id"": ""181913649"", ""text0"": ""My kid ...",https://i.imgflip.com/9o6x53.jpg,0,1,0,Variation 3 selected. Got stuck with Drake,None,2025-03-22T06:44:57.594058
7,8,2,Overengineering LLM Apps,181913649,"{""template_id"": ""181913649"", ""text0"": ""Buildin...",https://i.imgflip.com/9o6x7l.jpg,1,0,0,Variation 1 selected. Note the third option fa...,None,2025-03-22T06:46:31.867924
8,9,2,"I am trying to make a funny meme, but my sense...",181913649,"{""template_id"": ""181913649"", ""text0"": ""My atte...",https://i.imgflip.com/9o6xfz.jpg,0,0,1,Variation 2 selected. Got stuck on Drake,None,2025-03-22T06:47:01.208057
9,10,2,i want to represent that a builder of an app w...,181913649,"{""template_id"": ""181913649"", ""text0"": ""Clean, ...",https://i.imgflip.com/9o6xio.jpg,0,1,0,Variation 2 selected. Note that 2 of 3 failed ...,None,2025-03-22T06:47:37.871728
